# Chicago Crime - Predictive Modeling

Predict: crime hotspots (top locations), most frequent times (hour, day, month), and most common crime types.

Uses k-fold cross validation, model comparison, ensemble methods, and standard evaluation metrics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/processed/crimes_cleaned.csv')
print("Shape:", df.shape)
df.head(3)

## 1. Prepare Features and Targets

We define three prediction tasks:
- **Primary Type**: What kind of crime? (multi-class)
- **District**: Where (hotspot)? (multi-class)
- **Hour**: When (time of day)? (multi-class)

Sample data if large to keep training feasible.

In [ ]:
# Sample for faster training (use full data in production)
SAMPLE_SIZE = 100000
if len(df) > SAMPLE_SIZE:
    df = df.sample(n=SAMPLE_SIZE, random_state=42)

# Feature columns (numeric + encoded categorical)
num_features = ['Month', 'DayOfWeek', 'Quarter', 'WeekOfYear', 'IsWeekend', 'Arrest', 'Domestic']

# Encode Location Description (top 20 + other)
top_loc = df['Location Description'].value_counts().head(20).index.tolist()
df['Location_enc'] = df['Location Description'].apply(lambda x: x if x in top_loc else 'OTHER')
le_loc = LabelEncoder()
df['Location_enc'] = le_loc.fit_transform(df['Location_enc'].astype(str))

# Final feature set
feature_cols = num_features + ['Location_enc']
X = df[feature_cols].copy()

# Targets
y_type = df['Primary Type']
y_district = df['District'].astype(str)
y_hour = df['Hour']

print("Features:", feature_cols)
print("Targets: Primary Type, District, Hour")
X.head()

## 2. Model Comparison with K-Fold Cross Validation

Compare Logistic Regression, Random Forest, Gradient Boosting. Use 5-fold stratified CV with F1-weighted score.

In [ ]:
def run_cv_comparison(X, y, task_name):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    models = {
        'Logistic Regression': LogisticRegression(max_iter=500, random_state=42),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
        'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
    }
    
    results = {}
    for name, model in models.items():
        scores = cross_val_score(model, X_scaled, y, cv=skf, scoring='f1_weighted', n_jobs=-1)
        results[name] = {
            'mean_f1': scores.mean(),
            'std_f1': scores.std(),
            'scores': scores
        }
    
    print(f"\n--- {task_name} (5-fold CV F1-weighted) ---")
    for name, r in results.items():
        print(f"  {name}: {r['mean_f1']:.4f} (+/- {r['std_f1']:.4f})")
    return results, scaler

In [ ]:
# Task 1: Predict Primary Type
r1, scaler1 = run_cv_comparison(X, y_type, 'Predict Primary Type')

# Task 2: Predict District (top 10 for balance)
top_districts = y_district.value_counts().head(10).index.tolist()
mask_d = y_district.isin(top_districts)
X_d, y_d = X[mask_d], y_district[mask_d]
r2, scaler2 = run_cv_comparison(X_d, y_d, 'Predict District (top 10)')

# Task 3: Predict Hour (binned: morning/afternoon/evening/night for better balance)
def bin_hour(h):
    if 6 <= h < 12: return 0  # Morning
    if 12 <= h < 18: return 1  # Afternoon
    if 18 <= h < 24: return 2  # Evening
    return 3  # Night
y_hour_bin = df['Hour'].apply(bin_hour)
r3, scaler3 = run_cv_comparison(X, y_hour_bin, 'Predict Time-of-Day Bin (Morning/Afternoon/Evening/Night)')

## 3. Best Model Selection & Full Training

Train the best model per task on full train set and evaluate on hold-out test set.

In [ ]:
def train_eval_best(X, y, task_name):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)
    
    best = GradientBoostingClassifier(n_estimators=100, random_state=42)
    best.fit(X_tr_s, y_tr)
    y_pred = best.predict(X_te_s)
    
    print(f"\n=== {task_name} - Test Results ===")
    print("Accuracy:", accuracy_score(y_te, y_pred).round(4))
    print("F1 (weighted):", f1_score(y_te, y_pred, average='weighted').round(4))
    print("\nClassification Report:")
    print(classification_report(y_te, y_pred, zero_division=0))
    return best, y_pred, y_te

In [ ]:
_, _, _ = train_eval_best(X, y_type, 'Primary Type')

In [ ]:
_, _, _ = train_eval_best(X_d, y_d, 'District')

In [ ]:
_, _, _ = train_eval_best(X, y_hour_bin, 'Time-of-Day Bin')

## 4. Ensemble Model (Voting)

Combine Logistic Regression, Random Forest, Gradient Boosting via soft voting for Primary Type.

In [ ]:
from sklearn.ensemble import VotingClassifier

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y_type, test_size=0.2, random_state=42, stratify=y_type)

ensemble = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(max_iter=500, random_state=42)),
        ('rf', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
        ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42))
    ],
    voting='soft'
)
ensemble.fit(X_tr, y_tr)
y_pred_ens = ensemble.predict(X_te)

print("Ensemble (Voting) - Primary Type")
print("Accuracy:", accuracy_score(y_te, y_pred_ens).round(4))
print("F1 (weighted):", f1_score(y_te, y_pred_ens, average='weighted').round(4))

## 5. Feature Importance (Random Forest)

Which features drive predictions?

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_tr, y_tr)
imp = pd.DataFrame({'feature': feature_cols, 'importance': rf.feature_importances_})
imp = imp.sort_values('importance', ascending=False)
imp.plot(x='feature', y='importance', kind='barh', figsize=(8, 5), legend=False)
plt.title('Feature Importance (Primary Type)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## 6. Summary & Next Steps

- **Primary Type**: Multi-class classification; ensemble can slightly improve F1.
- **District (hotspots)**: Predicting top districts; spatial features (lat/lon bins) could help.
- **Time-of-day**: Binned prediction; raw hour prediction is more granular but noisier.
- **Production**: Use full data, add more features (lat/lon grids, lag features), tune hyperparameters.